In [ ]:
# Keep things clean - no annoying warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# All the stuff we need
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_absolute_error,precision_score,accuracy_score


In [ ]:
# Load the glass dataset
df=pd.read_csv("D:\Machine Learning\Datasets\Cases\Image_Segmentation\Image_Segmentation.csv")

,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [ ]:
# Convert glass types to numbers (sklearn needs numbers, not text)
le=LabelEncoder()
df['Class']=le.fit_transform(df['Class'])

# Split into features and target
X,y=df.drop('Class',axis=1),df['Class']

In [ ]:
# 70-30 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Time to find the best hyperparameters
# Testing different solvers (optimization methods)
solver=['liblinear', 'sag', 'lbfgs', 'newton-cg', 'saga', 'newton-cholesky']

# Try with and without L2 penalty
penalties=['l1',None]

# C controls regularization - testing 20 values from 0.0001 to 15
Cs=np.linspace(.0001,15,20)

# Store all results here
scores = []

# Loop through every combination - this might take a while
for s in solver:
    for p in penalties:
        for c in Cs:
            try:

                # Train model with current settings
                lr=LogisticRegression(penalty=p, solver=s, max_iter=1000, C=c)
                lr.fit(X_train, y_train)
                y_pred=lr.predict(X_test)

                # Save the accuracy
                scores.append([s, p, c, accuracy_score(y_test, y_pred)])
            except ValueError as e:

                # Some combinations don't work together - skip them
                print(f"Skipping solver={s}, penalty={p}, C={c:.4f}: {e}")
                scores.append([s, p, c, None])  # Include C value in error case
            
# Put results in a dataframe for easier viewing
df_scores=pd.DataFrame(scores,columns=('solver','penalty','C','score'))

# Display top results
print("Top 10 combinations:")
print(df_scores.sort_values('score', ascending=False).head(10))

# Display top results
print("Top 10 combinations:")
print(df_scores.sort_values('score', ascending=False).head(10))

# Find the best parameters
best_row = df_scores.loc[df_scores['score'].idxmax()]
print(f"\nBest combination:")
print(f"Solver: {best_row['solver']}")
print(f"Penalty: {best_row['penalty']}")
print(f"C: {best_row['C']:.4f}")
print(f"Accuracy: {best_row['score']:.4f}")


Skipping solver=liblinear, penalty=None, C=0.0001: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=0.7896: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=1.5790: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=2.3685: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=3.1580: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=3.9474: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=4.7369: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=5.5264: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=6.3158: penalty=None is not supported for the liblinear solver
Skipping solver=liblinear, penalty=None, C=7.1

In [ ]:
# Now train the final model with best settings on ALL data
test_model=LogisticRegression(solver='liblinear',penalty='l1',C=0.7896)
test_model.fit(X_train,y_train)

LogisticRegression(C=0.7896, penalty='l1', solver='liblinear')

In [ ]:
# Load test data and make predictions
tst=pd.read_csv("D:\Machine Learning\Datasets\Cases\Image_Segmentation\\tst_img.csv")
pred_class=test_model.predict(tst)

In [ ]:
# Convert predictions back to original image class names
tst['predicted_class']=le.inverse_transform(pred_class)

In [ ]:
# Displaying prediction
tst

,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean,predicted_class
0,22,90,10,0,0,0.666668,0.044444,0.880000,0.562963,112.000000,105.888885,128.555560,106.000000,-22.777779,45.222220,-22.444445,128.555560,0.179697,-2.097815,SKY
1,210,200,9,0,0,1.300000,0.998145,1.611111,1.123816,49.481480,45.000000,60.666668,43.000000,-14.111111,35.000000,-19.444445,60.666668,0.290788,-1.987599,PATH
2,240,184,9,0,0,0.500000,0.077778,0.777778,0.785185,11.851851,9.777778,9.888889,15.888889,-5.000000,-5.888889,13.000000,15.888889,0.500000,2.128646,GRASS
3,130,191,9,0,0,1.000000,0.400000,1.500000,1.011111,7.333334,5.333334,5.000000,11.222222,-7.000000,-5.666666,11.666667,11.222222,0.535820,2.122422,GRASS
